# NumPy vs Python Lists — Performance Comparison

## Why does this matter?

In data science and scientific computing, we routinely work with **millions of numbers** — pixel values in an image, stock prices, sensor readings, model weights, etc.

Python's built-in `list` is flexible and easy to use, but it was **not designed for heavy numeric computation**. NumPy (`numpy`) is a library built specifically for this purpose.

### In this notebook we compare Python lists vs NumPy arrays across four dimensions:

| # | Benchmark | What it tests |
|---|-----------|---------------|
| 1 | **Memory size** | How much RAM each uses |
| 2 | **Element-wise addition** | Adding a constant to every element |
| 3 | **Element-wise squaring** | Squaring every element |
| 4 | **Vector addition** | Adding two equal-length sequences |
| 5 | **Matrix multiplication** | Multiplying two 2D matrices |
| 6 | **Statistics** | Mean, sum, max, standard deviation |

### Key concepts you will see:
- **Vectorization**: NumPy applies operations to entire arrays at once — no Python `for` loop needed
- **Contiguous memory**: NumPy stores all elements in one block of memory, making operations cache-friendly and fast
- **C under the hood**: NumPy's operations are implemented in optimized C/Fortran code

---
## Setup — Import Libraries

In [1]:
import numpy as np
import time         # measure elapsed CPU time
import random       # generate random numbers for lists
import math         # for manual std deviation calculation
from sys import getsizeof  # inspect object size in memory

---
## Benchmark 1 — Memory Size

### Why does memory matter?
- Less memory → more data fits in RAM → no expensive disk I/O
- Smaller footprint → better CPU cache utilization → faster computation

### Why is a Python list larger?
A Python `list` stores **references (pointers)** to Python objects. Each integer in a list is a full Python object (~28 bytes each). A NumPy array stores **raw numeric values** in a compact, typed buffer (e.g., 8 bytes per `int64`).

```
Python list:  [ ptr → PyInt(0), ptr → PyInt(1), ptr → PyInt(2), ... ]   ← overhead per element
NumPy array:  [ 0, 1, 2, 3, 4, ... ]                                     ← raw bytes, no overhead
```

In [2]:
# Compare memory footprint of a Python list vs a NumPy array
# for the same 100 integers

n_elements = 100   # try changing to 100_000 and observe the gap widen

# --- Python list ---
x_list = list(range(0, n_elements))
print("size of list in memory :", getsizeof(x_list), "bytes")

# --- NumPy array ---
x_np = np.arange(0, n_elements)
print("size of numpy in memory:", getsizeof(x_np), "bytes")

size of list in memory : 856 bytes
size of numpy in memory: 912 bytes


### 📝 Takeaway
At small sizes the difference looks modest, but at scale (millions of elements) NumPy uses **~50% less memory** than an equivalent Python list. This is because `getsizeof` on a list only counts the list container + pointer array — the referenced integer objects add significant hidden overhead that NumPy avoids entirely.

---
## Benchmark 2 — Element-wise Addition

**Task**: Add `100` to every element of a 10-million-element sequence.

### Python list approach
Must loop over every element in Python — this means 10 million interpreter steps, each with function call overhead.

In [3]:
n_elements = 10_000_000  # 10 million elements
x_list = list(range(0, n_elements))
print("1st 10 numbers(Before):", x_list[:10])

start = time.process_time()

x_new = []
for e in x_list:          # Python for-loop: slow — one step per element
    x_new.append(e + 100)

end = time.process_time()

print("Time taken:", end - start)
print("1st 10 numbers(After):", x_new[:10])

1st 10 numbers(Before): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Time taken: 0.8125
1st 10 numbers(After): [100, 101, 102, 103, 104, 105, 106, 107, 108, 109]


### NumPy approach
The `+` operator is **overloaded** for NumPy arrays. Writing `x_numpy + 100` broadcasts the scalar `100` across every element in a single vectorized C call — no Python loop.

In [4]:
x_list  = list(range(0, n_elements))
x_numpy = np.array(x_list)

start = time.process_time()

x_new = x_numpy + 100     # vectorized: no Python loop, runs in compiled C

end = time.process_time()
print("Time taken:", end - start)

Time taken: 0.125


In [18]:
# Speedup factor = list time / numpy time
0.71875 / 0.109375

6.571428571428571

### 📝 Takeaway
NumPy is roughly **6–7× faster** for element-wise addition. The gap grows with array size.

---
## Benchmark 3 — Element-wise Squaring

**Task**: Square every element of a 30-million-element sequence.

Squaring is more compute-intensive than addition, so the advantage of NumPy's compiled code becomes even more pronounced.

In [3]:
# Python list — manual loop with exponentiation

n_elements = 30_000_000  # 30 million
x_list = list(range(0, n_elements))
print("First 10 numbers of list    :", x_list[:10])

start = time.process_time()

x_new = []
for i in x_list:          # 30 million Python iterations
    x_new.append(i ** 2)

end = time.process_time()
print("First 10 numbers of new list:", x_new[:10])
print("time taken:", end - start)

First 10 numbers of list    : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
First 10 numbers of new list: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
time taken: 5.015625


In [12]:
# NumPy — vectorized exponentiation with ** operator

x_list  = list(range(0, n_elements))
x_numpy = np.array(x_list)
print("First 10 numbers of numpy array    :", x_numpy[:10])

start = time.process_time()

x_new = x_numpy ** 2      # vectorized: entire array squared in one call

end = time.process_time()
print("First 10 numbers of new numpy array:", x_new[:10])
print("time taken:", end - start)

First 10 numbers of numpy array    : [0 1 2 3 4 5 6 7 8 9]
First 10 numbers of new numpy array: [ 0  1  4  9 16 25 36 49 64 81]
time taken: 0.015625


In [5]:
# Speedup factor
5.0 / 0.1

50.0

### 📝 Takeaway
NumPy is **~50× faster** for squaring. The heavier the computation per element, the more NumPy's compiled backend shines.

---
## Benchmark 4 — Vector Addition (Adding Two Sequences)

**Task**: Add two 10-million-element sequences element-by-element.

### Python list approach
Requires `zip()` to pair elements and a loop to accumulate results — slow.

In [14]:
n_elements = 10_000_000  # 10 million

v1_list = list(range(n_elements))
v2_list = list(range(n_elements))
print("vector1    :", v1_list[:10])
print("vector2    :", v2_list[:10])

start = time.process_time()

result_list = []
for x, y in zip(v1_list, v2_list):  # pairs elements, then adds — slow
    result_list.append(x + y)

end = time.process_time()
print("result_list:", result_list[:10])
print(end - start)

vector1    : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
vector2    : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
result_list: [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
0.96875


### NumPy approach
The `+` operator between two arrays performs **element-wise addition** via a single vectorized call. No `zip`, no loop.

In [15]:
arr1 = np.arange(n_elements)
arr2 = np.arange(n_elements)

start = time.process_time()

result_array = arr1 + arr2   # vectorized element-wise addition — one C call

end = time.process_time()
print(end - start)

0.015625


In [24]:
# Speedup factor
1.471 / 0.0350

42.028571428571425

### 📝 Takeaway
NumPy is **~42× faster** for vector addition. In machine learning, adding gradient vectors or weight updates happens millions of times during training — this speedup is critical.

---
## Benchmark 5 — Matrix Multiplication

**Task**: Multiply two 300×300 matrices (90,000 elements each).

Matrix multiplication is the **core operation of deep learning** — every layer of a neural network computes `output = weights @ input`. Making this fast is non-negotiable.

### Python list approach
Requires three nested loops — O(N³) Python operations. For N=300, that's 27 million iterations.

In [16]:
N = 300  # matrix dimensions: N×N  (try N=400 to see ~18 seconds)

A_list = [[random.randint(1, 10) for _ in range(N)] for _ in range(N)]
B_list = [[random.randint(1, 10) for _ in range(N)] for _ in range(N)]

start = time.process_time()

# Classic triple nested loop for matrix multiplication
result_list = [[0 for _ in range(N)] for _ in range(N)]
for i in range(N):           # row of A
    for j in range(N):       # column of B
        for k in range(N):   # dot product accumulation
            result_list[i][j] += A_list[i][k] * B_list[k][j]

end = time.process_time()
print(end - start)

3.453125


### NumPy approach
The `@` operator (or `np.matmul`) delegates to **BLAS/LAPACK** — highly optimized linear algebra routines that exploit CPU instruction sets and multi-core parallelism.

In [17]:
A_np = np.random.randint(1, 11, size=(N, N))
B_np = np.random.randint(1, 11, size=(N, N))

start = time.process_time()

result_np = A_np @ B_np   # @ is the matrix multiplication operator (Python 3.5+)
                           # equivalent: np.matmul(A_np, B_np)

end = time.process_time()
print(end - start)

0.0


In [27]:
# Speedup factor
13.59 / 0.1005

135.22388059701493

### 📝 Takeaway
NumPy is **~135× faster** for matrix multiplication. This is the single biggest reason deep learning frameworks (TensorFlow, PyTorch) are built on top of NumPy-style array operations.

---
## Benchmark 6 — Statistical Operations

**Task**: Compute **mean, sum, max, and standard deviation** of 10 million random integers.

Statistical aggregations are fundamental to data analysis, feature engineering, and model evaluation.

### Python list approach
Uses built-in functions (`sum`, `max`) which are reasonably fast, but the manual standard deviation still requires a list comprehension and multiple passes over the data.

In [18]:
SIZE = 10_000_000  # 10 million elements
x_list = [random.randint(1, 100) for _ in range(SIZE)]
print("1st 10 elements of x_list:", x_list[:10])

start = time.process_time()

# Built-in Python functions for sum, max
x_sum  = sum(x_list)
x_mean = x_sum / len(x_list)
x_max  = max(x_list)

# Standard deviation requires a manual two-pass calculation
mean_diff = [(x - x_mean) ** 2 for x in x_list]   # pass 1: squared deviations
x_std = math.sqrt(sum(mean_diff) / len(x_list))    # pass 2: average → sqrt

end = time.process_time()

print("Mean:", x_mean)
print("Sum:", x_sum)
print("Max:", x_max)
print("Standard Deviation:", x_std)
print("Time taken:", end - start, " seconds")

1st 10 elements of x_list: [23, 38, 28, 24, 46, 3, 61, 93, 61, 14]
Mean: 50.5102283
Sum: 505102283
Max: 100
Standard Deviation: 28.86379347178536
Time taken: 1.21875  seconds


### NumPy approach
NumPy provides built-in functions for all common statistics. They all run in compiled C with single-pass algorithms where possible.

In [19]:
x_array = np.random.randint(1, 100, SIZE)

start = time.process_time()

x_mean = np.mean(x_array)   # arithmetic mean
x_sum  = np.sum(x_array)    # total sum
x_max  = np.max(x_array)    # maximum value
x_std  = np.std(x_array)    # standard deviation — all in C, no Python loop

end = time.process_time()

print("Mean:", x_mean)
print("Sum:", x_sum)
print("Max:", x_max)
print("Standard Deviation:", x_std)
print("Time taken:", end - start, " seconds")

Mean: 50.0075041
Sum: 500075041
Max: 99
Standard Deviation: 28.57710755112356
Time taken: 0.0625  seconds


### 📝 Takeaway
NumPy is **~20× faster** for statistical operations. The syntax is also cleaner — one function call per statistic versus manual multi-step calculations.

---
## Summary

| Benchmark | Python List | NumPy Array | Speedup |
|-----------|------------|-------------|--------|
| Element-wise addition (10M) | ~0.72 s | ~0.11 s | **~6×** |
| Element-wise squaring (30M) | ~5.0 s | ~0.10 s | **~50×** |
| Vector addition (10M) | ~1.47 s | ~0.04 s | **~42×** |
| Matrix multiplication (300×300) | ~13.6 s | ~0.10 s | **~135×** |
| Statistics (10M) | ~1.22 s | ~0.06 s | **~20×** |

### Why is NumPy so much faster?

1. **No interpreter overhead** — operations run in compiled C/Fortran, bypassing the Python interpreter loop
2. **Contiguous memory layout** — all elements stored side-by-side; maximizes CPU cache hits
3. **Vectorized instructions (SIMD)** — modern CPUs can process multiple numbers in a single clock cycle; NumPy exploits this
4. **Optimized libraries** — matrix operations use BLAS/LAPACK, tuned specifically for numeric linear algebra
5. **No type boxing** — integers are raw bytes, not Python objects; no allocation/deallocation overhead

### When should you use Python lists instead?
- Mixed-type collections (e.g., `['Alice', 30, True]`)
- When elements are objects, not numbers
- When you frequently add/remove elements (dynamic resizing)
- Small collections where performance is not a concern

> **Rule of thumb**: If you're doing math on sequences of numbers → always use NumPy.